# Feast Feature Store on prokube

**Scenario:** You work at an online retailer. Your team wants to predict
whether a customer will return their next order. To do that, you need
customer-level features (order history, return rates, spending patterns)
available for both model training and real-time inference.

This notebook walks through the full Feast workflow:

1. **Setup** — install dependencies, configure the Feast client
2. **Generate data** — simulate customer order history
3. **Define features** — entities, feature views, and on-demand transformations
4. **Register** — push definitions to the Feast registry
5. **Train** — retrieve historical features, preprocess, train a return predictor
6. **Materialize** — push latest values to Redis for online serving
7. **Serve** — predict return risk for incoming orders in real time

Everything happens inline in this notebook — no external Python files or CLI
commands needed.

### Prerequisites

**Before running this notebook**, follow the setup steps in
[`README.md`](README.md) first. You need to create:

1. The **Redis password secret** (`redis-feast`)
2. The **Redis instance** (`redis-cr.yaml`) — wait until the pod is Running
3. The **Feast Redis secret** (`feast-redis-config`)
4. The **FeatureStore CR** (`feast-cr.yaml`) — wait until Ready

If any of these are missing, the notebook will fail at the "Configure the
Feast client" step.

> **Note:** This setup uses a SQLite registry on `/tmp` which does not
> survive pod restarts. For a production-ready deployment, see the
> [Production Setup](#Production-Setup) section at the end.


---
## 1. Setup


In [ ]:
!pip install -q 'feast[redis]' scikit-learn

### Configure the Feast client

We build `feature_store.yaml` dynamically by reading the Redis connection
string from the `feast-redis-config` secret.

**About the registry path (`/tmp/registry.db`):**
The registry is a small SQLite database that stores feature definitions
(entities, feature views, data sources). We write it to `/tmp` which means
it does **not** survive a pod restart. That's fine — the registry only holds
*definitions*, not data. Your actual feature *values* live in Redis (online
store, persistent) and parquet files (offline store, on PVC). Just re-run
the "Define & Register" cell after a restart to recreate it.


In [ ]:
import base64
import subprocess
import yaml


def get_namespace():
    """Read the current namespace from the pod's service account."""
    try:
        with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
            return f.read().strip()
    except FileNotFoundError:
        return subprocess.check_output(
            ["kubectl", "config", "view", "--minify", "-o", "jsonpath={..namespace}"]
        ).decode().strip()


def get_redis_connection_string():
    """Read the Redis connection string from the feast-redis-config secret."""
    result = subprocess.run(
        ["kubectl", "get", "secret", "feast-redis-config",
         "-o", "jsonpath={.data.redis}"],
        capture_output=True, text=True, check=True,
    )
    raw = base64.b64decode(result.stdout).decode()
    return yaml.safe_load(raw)["connection_string"]


NAMESPACE = get_namespace()
REDIS_CONNECTION_STRING = get_redis_connection_string()
FEAST_PROJECT = "retail_features"

feature_store_yaml = (
    f"project: {FEAST_PROJECT}\n"
    "provider: local\n"
    "offline_store:\n"
    "    type: file\n"
    "online_store:\n"
    "    type: redis\n"
    f"    connection_string: \"{REDIS_CONNECTION_STRING}\"\n"
    "registry:\n"
    "    registry_type: file\n"
    "    path: /tmp/registry.db\n"
    "auth:\n"
    "    type: no_auth\n"
    "entity_key_serialization_version: 3\n"
)

with open("feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("feature_store.yaml written")
print(f"Namespace: {NAMESPACE}")
print(f"Redis: {REDIS_CONNECTION_STRING.split(',')[0]}")


---
## 2. Generate sample data

We simulate a customer order history table — the kind of data your data
pipeline would produce daily. Each row represents the aggregated stats for
one customer at one point in time:

| Column | Meaning |
|--------|--------|
| `customer_id` | Unique customer identifier |
| `total_orders` | Total number of orders placed |
| `total_returns` | Total number of returned orders |
| `avg_order_value` | Average order value in EUR |
| `days_since_last_order` | Days since the customer's last order |
| `returned` | Did the customer return their most recent order? (label) |

In a real project, this would come from your data warehouse or ETL pipeline.


In [ ]:
import datetime
import os

import numpy as np
import pandas as pd

np.random.seed(42)
n_customers = 200
n_snapshots = 10  # 10 daily snapshots per customer
n = n_customers * n_snapshots
now = datetime.datetime.now()

customer_ids = np.repeat(np.arange(1, n_customers + 1), n_snapshots)
timestamps = []
for _ in range(n_customers):
    timestamps.extend([now - datetime.timedelta(days=i) for i in range(n_snapshots)])

# Simulate realistic customer stats
total_orders = np.random.randint(1, 80, n).astype(np.int64)
total_returns = np.array([
    np.random.binomial(orders, np.random.uniform(0.05, 0.4))
    for orders in total_orders
]).astype(np.int64)
avg_order_value = np.random.uniform(15.0, 250.0, n).astype(np.float32)
days_since_last_order = np.random.randint(0, 90, n).astype(np.int64)

# Label: customers with high return rates and high order values are more
# likely to return. Add noise to keep it realistic.
return_rate = total_returns / np.maximum(total_orders, 1)
return_prob = 0.3 * return_rate + 0.002 * avg_order_value / 250.0 + np.random.normal(0, 0.1, n)
returned = (return_prob > 0.15).astype(np.int64)

customer_df = pd.DataFrame({
    "customer_id": customer_ids,
    "event_timestamp": timestamps,
    "total_orders": total_orders,
    "total_returns": total_returns,
    "avg_order_value": avg_order_value,
    "days_since_last_order": days_since_last_order,
    "returned": returned,
    "created": timestamps,
})

os.makedirs("data", exist_ok=True)
customer_df.to_parquet("data/customer_orders.parquet")
print(f"Created {n} rows for {n_customers} customers ({n_snapshots} snapshots each)")
print(f"Return rate in dataset: {returned.mean():.1%}")
customer_df.head(10)


---
## 3. Define features

In Feast you define features as Python objects. There are two kinds:

- **FeatureView**: maps to columns in an existing data source (parquet file,
  database table, etc.). Feast stores and serves them — but doesn't compute
  anything. Your pipeline is responsible for producing the data.

- **On Demand Feature View (ODFV)**: a lightweight transformation that Feast
  executes at query time. It can combine existing features, add request-time
  inputs, or compute derived values. The transformation runs inline — during
  `get_historical_features()` and `get_online_features()` — so it's always
  consistent between training and serving.

Use FeatureViews for raw/precomputed data. Use ODFVs for derived features
that should be computed the same way everywhere.


In [ ]:
from datetime import timedelta

from feast import Entity, FeatureStore, FeatureView, Field, FileSource, ValueType
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float32, Float64, Int64

# ---------------------------------------------------------------------------
# Entity: the "primary key" for feature lookups.
# When you request features, you provide a customer_id.
# ---------------------------------------------------------------------------
customer = Entity(
    name="customer_id",
    value_type=ValueType.INT64,
    description="Unique customer identifier",
)

# ---------------------------------------------------------------------------
# Data source: points to the parquet file produced by the data pipeline.
# ---------------------------------------------------------------------------
customer_orders_source = FileSource(
    path="data/customer_orders.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

# ---------------------------------------------------------------------------
# FeatureView: declares which columns from the source are features.
# These are raw/precomputed values from your data pipeline.
#
# Note: the source data also contains "returned" (did the customer return
# their last order?). We don't include it here because it's our prediction
# target — at serving time, we don't know the answer yet. Labels live
# outside Feast and are joined only during training.
# ---------------------------------------------------------------------------
customer_order_stats = FeatureView(
    name="customer_order_stats",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="total_orders", dtype=Int64),
        Field(name="total_returns", dtype=Int64),
        Field(name="avg_order_value", dtype=Float32),
        Field(name="days_since_last_order", dtype=Int64),
    ],
    source=customer_orders_source,
    online=True,
)

# ---------------------------------------------------------------------------
# On Demand Feature View: derived features computed by Feast.
#
# return_rate:  what fraction of orders did this customer return?
# return_risk:  return_rate * avg_order_value — a simple risk score.
#               High return rate + expensive orders = high financial risk.
#
# These are computed on-the-fly during get_historical_features() and
# get_online_features(). You define the formula once; Feast guarantees
# the same logic runs in training and serving.
# ---------------------------------------------------------------------------
@on_demand_feature_view(
    sources=[customer_order_stats],
    schema=[
        Field(name="return_rate", dtype=Float64),
        Field(name="return_risk", dtype=Float64),
    ],
    mode="pandas",
)
def customer_risk_features(features_df: pd.DataFrame) -> pd.DataFrame:
    df = pd.DataFrame()
    df["return_rate"] = features_df["total_returns"] / features_df["total_orders"].clip(lower=1)
    df["return_risk"] = df["return_rate"] * features_df["avg_order_value"]
    return df


print("Feature definitions created (not yet registered).")


---
## 4. Register features

`store.apply()` writes all definitions to the registry. After this, Feast
knows which features exist, where the data comes from, and how derived
features are computed.


In [ ]:
store = FeatureStore(repo_path=".")

store.apply([
    customer,
    customer_orders_source,
    customer_order_stats,
    customer_risk_features,
])

print("Registered:")
for fv in store.list_feature_views():
    print(f"  FeatureView: {fv.name}")
for odfv in store.list_on_demand_feature_views():
    print(f"  OnDemandFeatureView: {odfv.name}")


---
## 5. Retrieve historical features and train a model

`get_historical_features()` performs a **point-in-time join**: for each
customer, it finds the most recent feature values *as of that timestamp*.
This prevents data leakage — you only see features that were available when
the event occurred.

Note how we request both raw features (`customer_order_stats:total_orders`)
and derived features (`customer_risk_features:return_rate`). The ODFV runs
automatically — no extra code needed.

> **Note:** Feast will show a `RuntimeWarning` that on-demand feature views
> are experimental and don't scale well for offline retrieval. For this
> notebook-sized dataset that's irrelevant. For large-scale training data
> (millions of rows), precompute heavy features in your pipeline and use a
> regular `FeatureView` instead.


In [ ]:
# Build a query: "give me features for these customers, as of right now."
# Each row says: I want to know the feature values for customer X at time T.
# Feast will find the most recent feature values that were available at that
# timestamp — this is the "point-in-time join" that prevents data leakage.

all_customer_ids = list(range(1, n_customers + 1))
query_timestamp = now  # "as of right now"

entity_df = pd.DataFrame({
    "customer_id": all_customer_ids,
    "event_timestamp": [query_timestamp] * len(all_customer_ids),
})

print(f"Querying features for {len(all_customer_ids)} customers...")

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_order_stats:total_orders",
        "customer_order_stats:total_returns",
        "customer_order_stats:avg_order_value",
        "customer_order_stats:days_since_last_order",
        "customer_risk_features:return_rate",   # <-- computed by Feast
        "customer_risk_features:return_risk",   # <-- computed by Feast
    ],
).to_df()

print(f"Retrieved {len(training_df)} rows.")
training_df.head(10)


### Preprocessing

Before training, we need to clean up the data:

1. **Join the label** — `returned` is our prediction target, not a feature.
   We deliberately keep it out of Feast because at serving time (when a
   customer places a new order) we don't know yet whether they will return
   it — that's what the model predicts. Labels typically come from a
   separate source (e.g. your data warehouse) and are joined only for
   training.
2. **Filter out new customers** — customers with fewer than 3 orders don't
   have enough history for reliable features. Feeding them into the model
   would add noise.
3. **Drop nulls** — any rows where Feast couldn't find matching features.
4. **Normalize** — scale numeric features so the model doesn't overweight
   high-magnitude columns like `avg_order_value`.


In [ ]:
from sklearn.preprocessing import StandardScaler

# --- 1. Join the label from the source data ---
# Get the most recent snapshot per customer to use as label
latest_labels = (
    customer_df
    .sort_values("event_timestamp")
    .groupby("customer_id")
    .last()[["returned"]]
    .reset_index()
)
training_df = training_df.merge(latest_labels, on="customer_id", how="left")

print(f"After joining labels: {len(training_df)} rows")

# --- 2. Filter out new customers (< 3 orders) ---
before = len(training_df)
training_df = training_df[training_df["total_orders"] >= 3].copy()
print(f"After filtering new customers (< 3 orders): {len(training_df)} rows (dropped {before - len(training_df)})")

# --- 3. Drop nulls ---
before = len(training_df)
training_df = training_df.dropna()
print(f"After dropping nulls: {len(training_df)} rows (dropped {before - len(training_df)})")

# --- 4. Normalize features ---
FEATURE_COLS = ["total_orders", "total_returns", "avg_order_value",
                "days_since_last_order", "return_rate", "return_risk"]
TARGET = "returned"

scaler = StandardScaler()
X = pd.DataFrame(
    scaler.fit_transform(training_df[FEATURE_COLS]),
    columns=FEATURE_COLS,
    index=training_df.index,
)
y = training_df[TARGET]

print(f"\nTraining set: {len(X)} samples, {len(FEATURE_COLS)} features")
print(f"Class balance: {y.mean():.1%} returns")
X.describe().round(2)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["kept", "returned"]))


---
## 6. Materialize features to Redis

Materialization copies the latest feature values from the offline store
(parquet) into Redis for low-latency online serving.

Note: only `FeatureView` data is materialized to Redis. The ODFV features
(`return_rate`, `return_risk`) are computed on-the-fly when you call
`get_online_features()` — Feast reads the raw values from Redis and applies
the transformation inline.

In production you would run this on a schedule (e.g. daily after your
pipeline updates the parquet files).


In [ ]:
from datetime import datetime, timedelta

store.materialize(
    start_date=datetime.now() - timedelta(days=30),
    end_date=datetime.now(),
)
print("Materialized to Redis.")


---
## 7. Online feature serving — predict return risk

A customer just placed an order. Your order service calls Feast to get
their features, feeds them into the model, and decides whether to flag
the order for proactive customer service.

The raw features (`total_orders`, etc.) come from Redis. The derived
features (`return_rate`, `return_risk`) are computed on-the-fly by the
ODFV — same formula as during training.


In [ ]:
# Simulate: these 3 customers just placed a new order
customers_with_new_orders = [{"customer_id": 5}, {"customer_id": 42}, {"customer_id": 137}]

online_features = store.get_online_features(
    features=[
        "customer_order_stats:total_orders",
        "customer_order_stats:total_returns",
        "customer_order_stats:avg_order_value",
        "customer_order_stats:days_since_last_order",
        "customer_risk_features:return_rate",
        "customer_risk_features:return_risk",
    ],
    entity_rows=customers_with_new_orders,
).to_dict()

print("Online features (from Redis):")
online_df = pd.DataFrame(online_features)
online_df


In [ ]:
# Run inference: predict return probability and flag high-risk orders
X_inference = pd.DataFrame(
    scaler.transform(online_df[FEATURE_COLS]),
    columns=FEATURE_COLS,
)

return_probabilities = model.predict_proba(X_inference)[:, 1]

RISK_THRESHOLD = 0.5

print("\n--- Return Risk Assessment ---")
for cid, prob in zip(online_df["customer_id"], return_probabilities):
    flag = "HIGH RISK" if prob > RISK_THRESHOLD else "low risk"
    print(f"  Customer {cid}: return probability = {prob:.1%} [{flag}]")


---
## Summary

| Step | API | What happens |
|------|-----|-------------|
| Define | Python objects (Entity, FeatureView, ODFV) | Declare what features exist and how derived ones are computed |
| Register | `store.apply([...])` | Write definitions to the registry (once per session) |
| Train | `store.get_historical_features()` | Point-in-time join from parquet; ODFVs run inline |
| Preprocess | pandas / sklearn | Filter, clean, normalize before training |
| Materialize | `store.materialize()` | Push latest raw values to Redis |
| Serve | `store.get_online_features()` | Sub-ms lookup from Redis + ODFV computed inline |

### FeatureView vs On Demand Feature View

| | FeatureView | On Demand Feature View |
|-|-------------|------------------------|
| Data | Precomputed in your pipeline | Computed by Feast at query time |
| Materialized to Redis? | Yes | No — computed on-the-fly at query time |
| Good for | Raw/heavy features | Lightweight derived features, request-time inputs |
| Consistency | You ensure pipeline runs | Feast guarantees same logic in training & serving |

### About the registry

The registry (`/tmp/registry.db`) stores only *definitions* — not feature
values. It is ephemeral in this notebook setup. If your pod restarts, re-run
the "Define & Register" cells. Your feature *data* in Redis and on PVC is
not affected.


---
## Production Setup

This notebook uses a **local, interactive workflow** — you define features
inline, register them with `store.apply()`, and connect directly to Redis.
That's great for experimentation. In production, the architecture looks
different:

### 1. Feature definitions live in Git

Instead of defining features in a notebook, you put them in a `features.py`
file in a Git repository. The Feast Operator clones the repo on startup and
runs `feast apply` automatically:

```yaml
# feast-cr.yaml (production)
spec:
  feastProject: retail_features
  feastProjectDir:
    git:
      url: https://github.com/your-org/feast-feature-repo
      ref: main  # or pin to a commit SHA
```

This means feature definitions are version-controlled, reviewed via PRs,
and automatically deployed when the pod starts.

### 2. The Feast Server exposes a Registry Server

Add `server: {}` to the registry config to expose it as a gRPC endpoint.
Notebooks and other clients can then read feature metadata remotely:

```yaml
# feast-cr.yaml (production)
spec:
  services:
    registry:
      local:
        server: {}  # exposes gRPC on port 6570
        persistence:
          store:
            type: sql
            secretRef:
              name: feast-registry-db  # PostgreSQL for production
```

### 3. Notebooks use the remote client

The Feast Operator creates a ConfigMap (`feast-<name>-client`) with the
client config. Instead of building `feature_store.yaml` manually, your
notebook just mounts it or copies it:

```python
# Production notebook — no local registry, no direct Redis
store = FeatureStore(repo_path=".")  # reads feature_store.yaml from ConfigMap

# get_online_features goes through the Feast Server (HTTP)
# get_historical_features goes through the Offline Server (Arrow Flight)
# Feature metadata comes from the Registry Server (gRPC)
```

### 4. Materialization runs as a CronJob

Instead of running `store.materialize()` from a notebook, you set up a
Kubernetes CronJob that runs on a schedule (e.g. daily after your ETL
pipeline refreshes the customer data). The Feast Operator can manage this
via the `batchEngine` config.

### 5. Use production-grade storage

The only component in this notebook that is **not** production-ready is
the SQLite registry on `/tmp`. Redis and the parquet offline store are
fine for production:

| Component | This notebook | Production |
|-----------|--------------------|-----------|
| Registry | SQLite on `/tmp` (ephemeral) | PostgreSQL (SQL-backed, persistent) |
| Online Store | Redis via OpsTree operator | Same — prokube manages this for you |
| Offline Store | Parquet on PVC | Same — or S3/MinIO for larger datasets |
| Feast Server | 1 replica | Multi-replica with HPA autoscaling |

### Architecture overview

```
  You (notebook)                    Feast Server Pod
  ┌──────────────┐                 ┌──────────────────────┐
  │ FeatureStore  │── online ──────▶│ Online Feature Server │──▶ Redis
  │ (remote       │── historical ──▶│ Offline Feature Server│──▶ Parquet / S3
  │  client)      │── metadata ────▶│ Registry Server       │──▶ PostgreSQL
  └──────────────┘                 └──────────────────────┘
                                          ▲
  CronJob ── materialize ─────────────────┘
  Git repo ── feast apply (on pod start) ─┘
```

For the full production deployment guide, see the
[Feast Production Deployment Topologies](https://docs.feast.dev/how-to-guides/production-deployment-topologies)
documentation.
